---
# **Lab: Transformers**
---

# ▶️ CUDA tools...

In [1]:
!nvidia-smi

Sun Feb 22 10:31:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2060      WDDM  |   00000000:01:00.0  On |                  N/A |
| 29%   37C    P0             33W /  190W |    5798MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# ✅ Transformers from scratch

**Clean Transformer (Encoder–Decoder) from scratch in PyTorch**

What this file is optimized for:
- readability over micro-optimizations
- clear mapping to the standard Transformer diagram:
  - Embedding + PosEnc → [EncoderBlock x L] → memory
  - Target Embedding + PosEnc → [DecoderBlock x L] → logits

Key design choices (teaching-friendly):
- Pre-norm residual blocks (stable + common in modern code)
- Masks use boolean convention: True = allowed, False = blocked
- Projection returns logits (use nn.CrossEntropyLoss)
- Minimal but complete: attention, FFN, encoder, decoder, full model

You can pair this model with any dataset that yields:
  - batch["encoder_input"] : LongTensor (B, S)
  - batch["decoder_input"] : LongTensor (B, T)   # shifted right (BOS + y[:-1])
  - batch["label"]         : LongTensor (B, T)   # y (including EOS), padded with PAD

and then train end-to-end

In [1]:
from __future__ import annotations

import math
from typing import Optional, Tuple, Dict, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ----------------------------
# Utilities: masks
# ----------------------------
def make_padding_mask(x: torch.Tensor, pad_id: int = 0) -> torch.Tensor:
    """
    x: (B, T) token ids
    returns mask: (B, 1, 1, T) where True means "keep" (not padding)
    """
    return (x != pad_id).unsqueeze(1).unsqueeze(2)

def make_causal_mask(t: int, device: torch.device) -> torch.Tensor:
    """
    returns mask: (1, 1, T, T) lower-triangular, True means "allowed"
    """
    return torch.tril(torch.ones(t, t, device=device, dtype=torch.bool)).unsqueeze(0).unsqueeze(0)

def create_masks(
    src: torch.Tensor,
    tgt: torch.Tensor,
    pad_id: int = 0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    src: (B, S)
    tgt: (B, T)  (decoder input tokens, usually shifted-right)
    returns:
      src_mask: (B, 1, 1, S)   (for encoder self-attn and decoder cross-attn keys)
      tgt_mask: (B, 1, T, T)   (padding + causal for decoder self-attn)
    """
    device = src.device
    src_mask = make_padding_mask(src, pad_id=pad_id)  # (B, 1, 1, S)

    tgt_pad = make_padding_mask(tgt, pad_id=pad_id)  # (B, 1, 1, T)
    causal = make_causal_mask(tgt.size(1), device=device)  # (1, 1, T, T)
    tgt_mask = tgt_pad & causal  # broadcast to (B, 1, T, T)

    return src_mask, tgt_mask

# ----------------------------
# Core layers
# ----------------------------

class InputEmbedding(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.embedding(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, seq_len: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))  # (1, seq_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, : x.size(1), :]
        return self.dropout(x)


class LayerNormalization(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta


class FeedForwardBlock(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.dropout(F.relu(self.fc1(x))))


class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        dh = q.size(-1)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(dh)  # (B,H,Tq,Tk)

        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = attn @ v
        return out, attn


class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

        self.attn = ScaledDotProductAttention(dropout=dropout)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        b, tq, _ = query.shape
        _, tk, _ = key.shape

        q = self.w_q(query)
        k = self.w_k(key)
        v = self.w_v(value)

        q = q.view(b, tq, self.n_heads, self.d_head).transpose(1, 2)  # (B,H,Tq,Dh)
        k = k.view(b, tk, self.n_heads, self.d_head).transpose(1, 2)  # (B,H,Tk,Dh)
        v = v.view(b, tk, self.n_heads, self.d_head).transpose(1, 2)  # (B,H,Tk,Dh)

        out, _ = self.attn(q, k, v, mask=mask)  # (B,H,Tq,Dh)
        out = out.transpose(1, 2).contiguous().view(b, tq, self.d_model)  # (B,Tq,D)

        return self.w_o(out)


class ResidualConnection(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.norm = LayerNormalization(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, sublayer) -> torch.Tensor:
        return x + self.dropout(sublayer(self.norm(x)))


class ProjectionLayer(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)  # logits


# ----------------------------
# Encoder / Decoder blocks
# ----------------------------
class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, self_attention: MultiHeadAttentionBlock, feed_forward: FeedForwardBlock, dropout: float):
        super().__init__()
        self.self_attention = self_attention
        self.feed_forward = feed_forward
        self.res1 = ResidualConnection(d_model, dropout)
        self.res2 = ResidualConnection(d_model, dropout)

    def forward(self, x: torch.Tensor, src_mask: Optional[torch.Tensor]) -> torch.Tensor:
        x = self.res1(x, lambda t: self.self_attention(t, t, t, mask=src_mask))
        x = self.res2(x, self.feed_forward)
        return x


class Encoder(nn.Module):
    def __init__(self, d_model: int, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(d_model)

    def forward(self, x: torch.Tensor, src_mask: Optional[torch.Tensor]) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)


class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        self_attention: MultiHeadAttentionBlock,
        cross_attention: MultiHeadAttentionBlock,
        feed_forward: FeedForwardBlock,
        dropout: float,
    ):
        super().__init__()
        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.feed_forward = feed_forward

        self.res1 = ResidualConnection(d_model, dropout)
        self.res2 = ResidualConnection(d_model, dropout)
        self.res3 = ResidualConnection(d_model, dropout)

    def forward(
        self,
        x: torch.Tensor,
        encoder_output: torch.Tensor,
        src_mask: Optional[torch.Tensor],
        tgt_mask: Optional[torch.Tensor],
    ) -> torch.Tensor:
        x = self.res1(x, lambda t: self.self_attention(t, t, t, mask=tgt_mask))
        x = self.res2(x, lambda t: self.cross_attention(t, encoder_output, encoder_output, mask=src_mask))
        x = self.res3(x, self.feed_forward)
        return x


class Decoder(nn.Module):
    def __init__(self, d_model: int, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(d_model)

    def forward(
        self,
        x: torch.Tensor,
        encoder_output: torch.Tensor,
        src_mask: Optional[torch.Tensor],
        tgt_mask: Optional[torch.Tensor],
    ) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)


# ----------------------------
# Full Transformer
# ----------------------------
class Transformer(nn.Module):
    def __init__(
        self,
        encoder: Encoder,
        decoder: Decoder,
        src_embed: InputEmbedding,
        tgt_embed: InputEmbedding,
        src_pos: PositionalEncoding,
        tgt_pos: PositionalEncoding,
        projection: ProjectionLayer,
    ):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos
        self.projection = projection

    def encode(self, src: torch.Tensor, src_mask: Optional[torch.Tensor]) -> torch.Tensor:
        x = self.src_pos(self.src_embed(src))
        return self.encoder(x, src_mask)

    def decode(
        self,
        encoder_output: torch.Tensor,
        src_mask: Optional[torch.Tensor],
        tgt: torch.Tensor,
        tgt_mask: Optional[torch.Tensor],
    ) -> torch.Tensor:
        x = self.tgt_pos(self.tgt_embed(tgt))
        return self.decoder(x, encoder_output, src_mask, tgt_mask)

    def project(self, x: torch.Tensor) -> torch.Tensor:
        return self.projection(x)


# ----------------------------
# Builder
# ----------------------------
def build_transformer(
    src_vocab_size: int,
    tgt_vocab_size: int,
    src_seq_len: int,
    tgt_seq_len: int,
    d_model: int = 256,
    n_layers: int = 2,
    n_heads: int = 4,
    dropout: float = 0.1,
    d_ff: int = 512,
) -> Transformer:
    src_embed = InputEmbedding(d_model, src_vocab_size)
    tgt_embed = InputEmbedding(d_model, tgt_vocab_size)
    src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
    tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)

    enc_layers = []
    for _ in range(n_layers):
        attn = MultiHeadAttentionBlock(d_model, n_heads, dropout)
        ffn = FeedForwardBlock(d_model, d_ff, dropout)
        enc_layers.append(EncoderBlock(d_model, attn, ffn, dropout))
    encoder = Encoder(d_model, nn.ModuleList(enc_layers))

    dec_layers = []
    for _ in range(n_layers):
        self_attn = MultiHeadAttentionBlock(d_model, n_heads, dropout)
        cross_attn = MultiHeadAttentionBlock(d_model, n_heads, dropout)
        ffn = FeedForwardBlock(d_model, d_ff, dropout)
        dec_layers.append(DecoderBlock(d_model, self_attn, cross_attn, ffn, dropout))
    decoder = Decoder(d_model, nn.ModuleList(dec_layers))

    proj = ProjectionLayer(d_model, tgt_vocab_size)

    model = Transformer(encoder, decoder, src_embed, tgt_embed, src_pos, tgt_pos, proj)

    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)

    return model


# ----------------------------
# Tiny synthetic dataset: copy task
# ----------------------------
class SyntheticCopyDataset(Dataset):
    """
    Copy task with BOS/EOS.

    Token ids:
      PAD = 0
      BOS = 1
      EOS = 2
      payload tokens in [3, vocab_size-1]

    For each sample:
      src:          length S (padded)
      decoder_input length T=S+1: BOS + src_payload
      label         length T=S+1: src_payload + EOS
    """

    def __init__(
        self,
        n_samples: int,
        vocab_size: int,
        src_len: int,
        pad_id: int = 0,
        bos_id: int = 1,
        eos_id: int = 2,
        min_payload_id: int = 3,
        seed: int = 0,
    ):
        super().__init__()
        if vocab_size <= min_payload_id:
            raise ValueError("vocab_size too small for PAD/BOS/EOS + payload tokens")

        self.n_samples = n_samples
        self.vocab_size = vocab_size
        self.src_len = src_len

        self.pad_id = pad_id
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.min_payload_id = min_payload_id

        g = torch.Generator().manual_seed(seed)
        # Generate random payload sequences (no PAD/BOS/EOS)
        self.payload = torch.randint(
            low=min_payload_id,
            high=vocab_size,
            size=(n_samples, src_len),
            generator=g,
            dtype=torch.long,
        )

    def __len__(self) -> int:
        return self.n_samples

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        src = self.payload[idx]  # (S,)

        # decoder_input: BOS + src
        decoder_input = torch.empty(self.src_len + 1, dtype=torch.long)
        decoder_input[0] = self.bos_id
        decoder_input[1:] = src

        # label: src + EOS
        label = torch.empty(self.src_len + 1, dtype=torch.long)
        label[:-1] = src
        label[-1] = self.eos_id

        return {
            "encoder_input": src,
            "decoder_input": decoder_input,
            "label": label,
        }


# ----------------------------
# Training loop (works end-to-end)
# ----------------------------
def train_end_to_end(
    model: Transformer,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    device: torch.device,
    pad_id: int = 0,
    steps: int = 300,
    print_every: int = 50,
):
    model.train()
    it = iter(dataloader)

    for step in range(1, steps + 1):
        try:
            batch = next(it)
        except StopIteration:
            it = iter(dataloader)
            batch = next(it)

        src = batch["encoder_input"].to(device)   # (B,S)
        tgt = batch["decoder_input"].to(device)   # (B,T)
        labels = batch["label"].to(device)        # (B,T)

        src_mask, tgt_mask = create_masks(src, tgt, pad_id=pad_id)

        enc = model.encode(src, src_mask)
        dec = model.decode(enc, src_mask, tgt, tgt_mask)
        logits = model.project(dec)  # (B,T,V)

        loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if step % print_every == 0 or step == 1:
            print(f"step {step:04d}/{steps} | loss={loss.item():.4f}")


@torch.no_grad()
def greedy_decode_copy(
    model: Transformer,
    src: torch.Tensor,
    max_new_tokens: int,
    pad_id: int,
    bos_id: int,
    eos_id: int,
) -> torch.Tensor:
    """
    Simple greedy decoding for the synthetic copy task.
    src: (B,S)
    returns generated token ids (B, <=max_new_tokens)
    """
    device = src.device
    model.eval()

    src_mask = make_padding_mask(src, pad_id=pad_id)
    enc = model.encode(src, src_mask)

    B = src.size(0)
    ys = torch.full((B, 1), bos_id, dtype=torch.long, device=device)  # start with BOS

    for _ in range(max_new_tokens):
        tgt_mask = make_padding_mask(ys, pad_id=pad_id) & make_causal_mask(ys.size(1), device=device)
        dec = model.decode(enc, src_mask, ys, tgt_mask)
        logits = model.project(dec)[:, -1, :]  # (B,V)
        next_tok = torch.argmax(logits, dim=-1, keepdim=True)  # (B,1)
        ys = torch.cat([ys, next_tok], dim=1)
        if torch.all(next_tok.squeeze(1) == eos_id):
            break

    return ys



### training...

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Special tokens
PAD, BOS, EOS = 0, 1, 2

# Tiny settings for quick sanity check
VOCAB = 64
SRC_LEN = 16
TGT_LEN = SRC_LEN + 1  # BOS+src predicts src+EOS

model = build_transformer(
    src_vocab_size=VOCAB,
    tgt_vocab_size=VOCAB,
    src_seq_len=SRC_LEN,
    tgt_seq_len=TGT_LEN,
    d_model=128,
    n_layers=2,
    n_heads=4,
    dropout=0.1,
    d_ff=256,
).to(device)

# Synthetic data
dataset = SyntheticCopyDataset(
    n_samples=2000,
    vocab_size=VOCAB,
    src_len=SRC_LEN,
    pad_id=PAD,
    bos_id=BOS,
    eos_id=EOS,
    seed=0,
)

loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
# ignore_index not really needed here (no PAD in labels), but kept for completeness
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.0)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# Train quickly
train_end_to_end(
    model=model,
    dataloader=loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    pad_id=PAD,
    steps=400,
    print_every=50,
)

# Quick qualitative test
batch = next(iter(loader))
src = batch["encoder_input"][:4].to(device)

gen = greedy_decode_copy(
    model=model,
    src=src,
    max_new_tokens=TGT_LEN,  # generate up to BOS+SRC+EOS length
    pad_id=PAD,
    bos_id=BOS,
    eos_id=EOS,
)

print("\n--- Copy sanity check (token IDs) ---")
print("SRC:        ", src.cpu().tolist())
print("GENERATED:  ", gen.cpu().tolist())
print("EXPECTED:   ", [[BOS] + s + [EOS] for s in src.cpu().tolist()])

Using device: cuda
Model parameters: 684,608
step 0001/400 | loss=4.8309
step 0050/400 | loss=3.9991
step 0100/400 | loss=3.5988
step 0150/400 | loss=3.2279
step 0200/400 | loss=2.7392
step 0250/400 | loss=2.2546
step 0300/400 | loss=1.7762
step 0350/400 | loss=1.2767
step 0400/400 | loss=0.9779

--- Copy sanity check (token IDs) ---
SRC:         [[24, 14, 62, 29, 35, 47, 31, 5, 6, 23, 37, 47, 34, 15, 11, 63], [36, 20, 55, 12, 51, 20, 62, 42, 52, 38, 49, 42, 54, 29, 56, 44], [14, 56, 17, 24, 35, 4, 12, 47, 19, 19, 6, 8, 8, 40, 10, 12], [13, 15, 23, 63, 47, 21, 7, 49, 47, 28, 49, 57, 3, 36, 29, 61]]
GENERATED:   [[1, 24, 14, 62, 29, 35, 47, 31, 5, 6, 23, 37, 47, 34, 15, 11, 63, 2], [1, 36, 20, 55, 12, 51, 20, 62, 42, 52, 38, 49, 42, 54, 29, 56, 44, 2], [1, 14, 56, 17, 24, 35, 4, 12, 47, 19, 6, 19, 8, 8, 40, 10, 12, 2], [1, 13, 15, 23, 63, 47, 21, 7, 49, 47, 28, 57, 49, 3, 36, 29, 61, 2]]
EXPECTED:    [[1, 24, 14, 62, 29, 35, 47, 31, 5, 6, 23, 37, 47, 34, 15, 11, 63, 2], [1, 36, 20, 55, 

# Pretrained models (HggFace)

**Loading a Pretrained Translation Model**

We use HuggingFace to load a pretrained **English → French** Transformer.

```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig

model_name = "Helsinki-NLP/opus-mt-en-fr"
config = AutoConfig.from_pretrained(model_name)
config.tie_word_embeddings = False
```

- What this does:
	-	AutoConfig loads model architecture settings
	-	AutoTokenizer will convert text → tokens
	-	AutoModelForSeq2SeqLM loads an encoder–decoder Transformer
	-	tie_word_embeddings=False avoids a warning about shared embeddings

- This model is trained for machine translation.



**Tokenization and Generation**


```python
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tok("Studying deep learning is fun.", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=60)
```

- Full Pipeline Summary
	1.	Text → tokens
	2.	Encoder builds representation of input
	3.	Decoder generates translation step by step
	4.	Softmax produces probability over vocabulary
	5.	Tokens → translated sentence


> pip install transformers torch pillow requests sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig

model_name = "Helsinki-NLP/opus-mt-en-fr"
config = AutoConfig.from_pretrained(model_name)
config.tie_word_embeddings = False  # tell HF: don't try to tie them

tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tok("Studying deep learning is fun.", return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=60)
print(tok.decode(outputs[0], skip_special_tokens=True))